In [ ]:
!pip install -q ultralytics opencv-python matplotlib ipywidgets gradio

In [ ]:
import os
import time
from ultralytics import YOLO

# 1. Train / Val / Test Ayrımı İçin Özel YAML Yapılandırması (Bölüm 4)
voc_split_yaml = """
path: ../datasets/VOC
train:
  - images/train2012
  - images/train2007
  - images/val2012
val:
  - images/val2007
test:
  - images/test2007

names:
  0: aeroplane
  1: bicycle
  2: bird
  3: boat
  4: bottle
  5: bus
  6: car
  7: cat
  8: chair
  9: cow
  10: diningtable
  11: dog
  12: horse
  13: motorbike
  14: person
  15: pottedplant
  16: sheep
  17: sofa
  18: train
  19: tvmonitor
"""

# Önce standart VOC.yaml ile veri setinin otomatik inmesini sağlıyoruz,
# ardından kendi Train/Val/Test ayrımımızı kullanıyoruz.
with open("voc_custom.yaml", "w") as f:
  f.write(voc_split_yaml.strip())

# 2. Model Seçimi ve Eğitim (Bölüm 3 & 5)
model = YOLO("yolov8s.pt")

print("Veri seti kontrol ediliyor ve eğitim başlatılıyor...")
start_train = time.time()

# Not: İlk çalışmada VOC.yaml veri setini otomatik indirir (~2 GB).
train_results = model.train(
    data="VOC.yaml",
    epochs=20,  # Hızlı test için 10-20 arası tutabilirsin
    batch=16,
    imgsz=640,
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    project="CV_Odev_YOLO",
    name="yolov8s_voc",
    exist_ok=True,
    seed=42,
)

train_duration_min = (time.time() - start_train) / 60

# Eğitilen en iyi modeli yükle
best_model_path = "CV_Odev_YOLO/yolov8s_voc/weights/best.pt"
best_model = YOLO(best_model_path)

# 3. Test Kümesi Üzerinde Performans Değerlendirmesi (Bölüm 6)
print("\nTest kümesi üzerinde metrikler hesaplanıyor...")
metrics = best_model.val(data="voc_custom.yaml", split="test", iou=0.5)

mp = metrics.box.mp  # Mean Precision
mr = metrics.box.mr  # Mean Recall
map50 = metrics.box.map50  # mAP@0.50 (IoU=0.50)
map50_95 = metrics.box.map  # mAP@0.50:0.95
f1_score = (2 * mp * mr) / (mp + mr + 1e-8)
inference_speed_ms = metrics.speed["inference"]

print("\n" + "=" * 50)
print("BÖLÜM 6: PERFORMANS DEĞERLENDİRME SONUÇLARI")
print("=" * 50)
print(f"Precision (Kesinlik)      : {mp:.4f}")
print(f"Recall (Duyarlılık)       : {mr:.4f}")
print(f"F1-Score                  : {f1_score:.4f}")
print(f"mAP@0.50 (IoU=0.5)        : {map50:.4f}")
print(f"mAP@0.50:0.95             : {map50_95:.4f}")
print(f"Tahmin Süresi (Inference) : {inference_speed_ms:.2f} ms / görüntü")
print(f"Toplam Eğitim Süresi      : {train_duration_min:.2f} dakika")
print("=" * 50)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Veri seti kontrol ediliyor ve eğitim başlatılıyor...
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VOC.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, for

FileNotFoundError: [Errno 2] No such file or directory: 'CV_Odev_YOLO/yolov8s_voc/weights/best.pt'

In [ ]:
# @title 🚀 YOLOv8 İnteraktif Nesne Tespiti Uygulaması { display-mode: "form" }
import os
import time

try:
  import gradio as gr
  import ultralytics
except ImportError:
  os.system("pip install -q ultralytics gradio opencv-python")
  import gradio as gr

import cv2
from ultralytics import YOLO

# Eğitilen best.pt varsa onu, yoksa GitHub'dan veya standart ağırlıktan yükle
MODEL_PATH = "CV_Odev_YOLO/yolov8s_voc/weights/best.pt"
if not os.path.exists(MODEL_PATH):
  MODEL_PATH = "best.pt" if os.path.exists("best.pt") else "yolov8s.pt"

model_app = YOLO(MODEL_PATH)


def detect_objects(image, conf_threshold, iou_threshold):
  if image is None:
    return None, "Lütfen bir fotoğraf yükleyin."

  t0 = time.time()
  results = model_app.predict(
      source=image,
      conf=conf_threshold,
      iou=iou_threshold,
      imgsz=640,
      verbose=False,
  )
  dt_ms = (time.time() - t0) * 1000

  res = results[0]
  # Çıktı görüntüsünü hazırla (ultralytics BGR döner, RGB'ye çeviriyoruz)
  annotated_rgb = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)

  # Tespit edilen nesnelerin özet tablosu
  boxes = res.boxes
  md_text = f"### ⚡ Tahmin Süresi: `{dt_ms:.1f} ms` | Tespit Sayısı: `{len(boxes)}`\n\n"
  if len(boxes) > 0:
    md_text += "| # | Sınıf (Class) | Güven Skoru (Confidence) | Kutu Koordinatları |\n|---|---|---|---|\n"
    for i, box in enumerate(boxes, 1):
      cls_name = model_app.names[int(box.cls[0])]
      conf = float(box.conf[0])
      coords = [round(x, 1) for x in box.xyxy[0].tolist()]
      md_text += f"| {i} | **{cls_name}** | `{conf:.2%}` | `{coords}` |\n"
  else:
    md_text += "*Bu eşik değerinde nesne tespit edilemedi.*"

  return annotated_rgb, md_text


# Colab Hücresi İçinde Açılan Arayüz
with gr.Blocks(theme=gr.themes.Soft()) as demo:
  gr.Markdown("## 🎯 YOLOv8 Nesne Tespiti - İnteraktif Test Arayüzü")
  with gr.Row():
    with gr.Column():
      img_input = gr.Image(
          type="numpy", label="1. Veri Girişi: Fotoğraf Yükle veya Sürükle"
      )
      conf_slider = gr.Slider(
          0.05, 0.95, value=0.25, step=0.05, label="Confidence Threshold"
      )
      iou_slider = gr.Slider(
          0.10, 0.90, value=0.45, step=0.05, label="NMS IoU Threshold"
      )
      run_btn = gr.Button("🔍 Nesneleri Tespit Et", variant="primary")
    with gr.Column():
      img_output = gr.Image(
          type="numpy", label="2. Model Çıktısı: Tespit Edilen Nesneler"
      )
      tbl_output = gr.Markdown(label="Tespit Detayları")

  run_btn.click(
      fn=detect_objects,
      inputs=[img_input, conf_slider, iou_slider],
      outputs=[img_output, tbl_output],
  )

# inline=True sayesinde doğrudan Colab hücresinin içinde açılır
demo.launch(inline=True, share=True, height=650)

/tmp/ipykernel_4353/3601806632.py:58: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://77426254b0b7fc5431.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
